# AC nnU-Net v2 Training (Fold 0, 100 Epochs, T4 x2)

- Dataset: `Dataset502_AC` (642 AC training cases with filled ellipse masks)
- Split: Fixed patient-grouped 5-fold split (`splits_final.json`, Fold 0: 500 train / 142 val)
- Configuration: 2D (`nnUNetPlans__2d`)
- Trainer: `nnUNetTrainer_100epochs`
- Output: Preserves validation masks, `summary.json`, `progress.png`, and execution metrics.

In [ ]:
!nvidia-smi
!pip install --upgrade pip
# Install nnunetv2 (offline fallback if wheels present in dataset502-ac, otherwise PyPI matching HC18)
!pip install --no-index --find-links /kaggle/input/dataset502-ac/wheels nnunetv2 || pip install nnunetv2

In [ ]:
import shutil

nnunet_train_cmd = shutil.which("nnUNetv2_train")
if not nnunet_train_cmd:
    raise RuntimeError(
        "CRITICAL ERROR: nnUNetv2_train command was not found in PATH after installation! "
        "Halting execution immediately to avoid running silently for nothing."
    )
print(f"SUCCESS: Verified nnUNetv2_train binary found at: {nnunet_train_cmd}")

In [ ]:
import os
import shutil
from pathlib import Path

# Setup nnU-Net v2 directory paths and environment variables
os.environ["nnUNet_raw"] = "/kaggle/working/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/kaggle/working/nnUNet_results"

raw_dir = Path("/kaggle/working/nnUNet_raw")
prep_dir = Path("/kaggle/working/nnUNet_preprocessed")
res_dir = Path("/kaggle/working/nnUNet_results")
out_dir = Path("/kaggle/working/out")

raw_dir.mkdir(parents=True, exist_ok=True)
prep_dir.mkdir(parents=True, exist_ok=True)
res_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)
print("nnU-Net v2 directories created successfully.")

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

dataset_dir = raw_dir / "Dataset502_AC"

print("Locating dataset in /kaggle/input...")
# Check for dataset.json anywhere in /kaggle/input
dataset_jsons = list(Path("/kaggle/input").rglob("dataset.json"))
if dataset_jsons:
    src_dir = dataset_jsons[0].parent
    print(f"Found dataset directory at: {src_dir}")
    shutil.copytree(src_dir, dataset_dir, dirs_exist_ok=True)
else:
    # Check for zip archive anywhere in /kaggle/input
    zips = list(Path("/kaggle/input").rglob("*.zip"))
    if zips:
        print(f"Extracting {zips[0]} to {raw_dir}...")
        with zipfile.ZipFile(zips[0], "r") as zf:
            zf.extractall(raw_dir)
    else:
        print("Directory listing of /kaggle/input:")
        for root, dirs, files in os.walk("/kaggle/input"):
            print(f"  {root}: dirs={dirs[:5]}, files={files[:5]}")

assert (dataset_dir / "dataset.json").is_file(), f"Missing dataset.json in {dataset_dir}"
assert (dataset_dir / "imagesTr").is_dir(), f"Missing imagesTr in {dataset_dir}"
assert (dataset_dir / "labelsTr").is_dir(), f"Missing labelsTr in {dataset_dir}"
n_images = len(list((dataset_dir / "imagesTr").glob("*.png")))
n_labels = len(list((dataset_dir / "labelsTr").glob("*.png")))
print(f"Dataset unpacked successfully: {n_images} imagesTr, {n_labels} labelsTr.")
assert n_images == 642, f"Expected 642 imagesTr, found {n_images}"
assert n_labels == 642, f"Expected 642 labelsTr, found {n_labels}"

In [ ]:
!nnUNetv2_plan_and_preprocess -d 502 -c 2d --verify_dataset_integrity

In [ ]:
import json
import shutil

src_split = dataset_dir / "splits_final.json"
dst_split = prep_dir / "Dataset502_AC" / "splits_final.json"

assert src_split.is_file(), f"Source splits_final.json not found in {dataset_dir}"
dst_split.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src_split, dst_split)
print(f"Copied {src_split} -> {dst_split}")

with open(dst_split, "r") as f:
    splits = json.load(f)

print(f"Total folds in splits_final.json: {len(splits)}")
fold0 = splits[0]
n_train = len(fold0["train"])
n_val = len(fold0["val"])
print(f"Fold 0 Train cases: {n_train}")
print(f"Fold 0 Val cases  : {n_val}")
print(f"Total cases       : {n_train + n_val}")

assert n_train == 500, f"Expected 500 train cases, got {n_train}"
assert n_val == 142, f"Expected 142 val cases, got {n_val}"
overlap = set(fold0["train"]).intersection(set(fold0["val"]))
assert len(overlap) == 0, f"Patient leakage detected: {overlap}"
print("Diff check passed: exactly 500 train / 142 val cases with zero overlap.")

In [ ]:
import time

start_train = time.time()
print("=" * 70)
print("Starting AC nnU-Net training: 502 2d 0 -tr nnUNetTrainer_100epochs")
print("=" * 70)

!nnUNetv2_train 502 2d 0 -tr nnUNetTrainer_100epochs

elapsed = time.time() - start_train
print(f"AC nnU-Net training finished in {elapsed:.1f} s ({elapsed / 60:.1f} min)")

In [ ]:
trainer_dir = res_dir / "Dataset502_AC" / "nnUNetTrainer_100epochs__nnUNetPlans__2d" / "fold_0"

if trainer_dir.exists():
    print(f"Preserving results from {trainer_dir} to {out_dir}...")
    for pattern in ["summary.json", "progress.png", "training_log_*.txt"]:
        for match in trainer_dir.glob(pattern):
            shutil.copy2(match, out_dir / match.name)
            print(f"  Saved {match.name}")

    val_dir = trainer_dir / "validation"
    if val_dir.exists():
        dst_val = out_dir / "validation_fold0"
        shutil.copytree(val_dir, dst_val, dirs_exist_ok=True)
        n_val_files = len(list(dst_val.glob("*")))
        print(f"  Saved validation folder with {n_val_files} files.")

    manifest = {
        "dataset": "Dataset502_AC",
        "configuration": "2d",
        "fold": 0,
        "trainer": "nnUNetTrainer_100epochs",
        "train_cases": 500,
        "val_cases": 142,
        "elapsed_seconds": round(elapsed, 1),
    }
    with open(out_dir / "nnunet_run_summary.json", "w") as f:
        json.dump(manifest, f, indent=2)
    print("Summary manifest saved to", out_dir / "nnunet_run_summary.json")
else:
    print(f"WARNING: trainer directory {trainer_dir} does not exist!")